# User Preference Profile Construction

This notebook derives structured user preference profiles from the enriched purchase histories.

Instead of directly providing raw purchase histories to the language model, user behaviour is summarised into compact semantic features.

These profiles will later be used by an instruction-tuned language model to generate synthetic natural language shopping queries.

In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

In [2]:
PROJECT_DIR= Path.home()/"query-conditioned-recommender"
PROCESSED= PROJECT_DIR/"data"/"processed"
DATASET= PROCESSED/"enriched_user_histories.parquet"
OUTPUT= PROCESSED/"user_preference_profiles.parquet"

In [3]:
dataset = pd.read_parquet(DATASET)
dataset.head()

,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values,history_titles,history_categories,history_descriptions,ground_truth_title,ground_truth_categories,ground_truth_description
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B075B5T7L5, B00WA4PSEM, B09VS4V18K, B00G2UTVH...",B007VKII6A,5,"[2015-09-01T12:14:16.000000, 2015-09-01T12:23:...","[5, 5, 5, 5, 5]",2017-05-21 14:36:52.000,2,None,None,None,[Patriot LX Series 64GB High Speed Micro SDXC ...,"[[Electronics, Computers & Accessories, Comput...",[[Patriot’s LX Series UHS-I compatible MicroSD...,IDANCE CRAZY101 - Watt - Channel Recording Stu...,"[Electronics, Headphones, Earbuds & Accessorie...",[Super Aticulated Headband. Extra Bass with Op...
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00WA4PSEM, B09VS4V18K, B00G2UTVH0, B07RS68PJ...",B07454F4JH,5,"[2015-09-01T12:23:56.000000, 2016-10-02T21:27:...","[5, 5, 5, 5, 2]",2017-08-24 17:49:29.844,5,None,None,None,[The AquaAudio Cubo – Waterproof Bluetooth Wir...,"[[Electronics, Portable Audio & Video, Portabl...",[[AquaAudio Cubo Bluetooth Wireless Speaker wi...,Seagate Backup Plus Portable 5TB External Hard...,"[Electronics, Computers & Accessories, Data St...","[Looking for easy, on-the-go storage? Seagate ..."
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B09VS4V18K, B00G2UTVH0, B07RS68PJ7, B007VKII6...",B00SMVLPIK,5,"[2016-10-02T21:27:51.000000, 2016-10-02T22:15:...","[5, 5, 5, 2, 5]",2017-08-24 17:57:36.638,5,None,None,None,[Western Digital 2TB WD Black Performance Inte...,"[[Electronics, Computers & Accessories, Data S...",[[WD Black performance storage is designed to ...,Panasonic FOLDZ On-Ear Stereo Headphones with ...,"[Electronics, Headphones, Earbuds & Accessorie...",[The Panasonic FOLDZ RP-DJS150M combines the h...
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00G2UTVH0, B07RS68PJ7, B007VKII6A, B07454F4J...",B0179DDIQI,5,"[2016-10-02T22:15:43.000000, 2016-10-23T09:32:...","[5, 5, 2, 5, 5]",2017-08-24 18:06:02.421,5,None,None,None,[EBL Pack of 6 1000mAh NI-MH Home Phone Rechar...,"[[Electronics, Accessories & Supplies, Telepho...","[[features:, Chemistry: Ni-MH Voltage: 2.4V C...",BienSound HW50 Stereo Folding Headsets Strong ...,"[Electronics, Headphones, Earbuds & Accessorie...",[]
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B07RS68PJ7, B007VKII6A, B07454F4JH, B00SMVLPI...",B00X088DZI,5,"[2016-10-23T09:32:31.000000, 2017-05-21T14:36:...","[5, 2, 5, 5, 5]",2019-05-11 17:22:30.699,5,None,None,None,"[CyberPower CP900AVR AVR UPS System, 900VA/560...","[[Electronics, Computers & Accessories, Comput...",[[A mini-tower UPS with line interactive topol...,Kastar 5-Pack HHR-P105 Type 31 Battery Replace...,"[Electronics, Accessories & Supplies, Telephon...",[Bullet Point 1 Kastar Brand Replacement NI-MH...


In [4]:
def main_category(categories):
    result= []
    for cat in categories:
        if cat is None:
            result.append(None)
        elif len(cat) > 0:
            result.append(cat[-1])
        else:
            result.append(None)
    return result

In [5]:
dataset["history_main_categories"]= dataset["history_categories"].progress_apply(main_category)

100%|██████████| 5559/5559 [00:00<00:00, 400160.23it/s]


In [6]:
def preferred_categories(categories):
    cats = [c for c in categories if c is not None]
    if len(cats) == 0:
        return []
    counter= Counter(cats)
    return [c for c, _ in counter.most_common(3)]

In [7]:
dataset["preferred_categories"]= dataset["history_main_categories"].progress_apply(preferred_categories)

100%|██████████| 5559/5559 [00:00<00:00, 122049.09it/s]


In [8]:
dataset["average_rating"]= dataset["history_ratings"].progress_apply(np.mean)

100%|██████████| 5559/5559 [00:00<00:00, 142908.76it/s]


In [9]:
def liked_products(row):
    liked= []
    for title, rating in zip(
        row["history_titles"],
        row["history_ratings"]
    ):
        if rating >= 4:
            liked.append(title)
    return liked

In [10]:
dataset["liked_products"]= dataset.progress_apply(liked_products,axis=1)

100%|██████████| 5559/5559 [00:00<00:00, 102520.05it/s]


In [11]:
dataset["recent_products"]= dataset["history_titles"].progress_apply(lambda x: x[-2:])

100%|██████████| 5559/5559 [00:00<00:00, 1117261.77it/s]


In [12]:
def category_diversity(categories):
    return len(set(c for c in categories if c is not None))

dataset["category_diversity"]= dataset["history_main_categories"].progress_apply(category_diversity)

100%|██████████| 5559/5559 [00:00<00:00, 704776.95it/s]


In [13]:
def purchase_span(timestamps):
    if timestamps is None or len(timestamps) < 2:
        return 0
    ts= pd.to_datetime(timestamps)
    return (ts.iloc[-1]-ts.iloc[0]).days if hasattr(ts, "iloc") else (ts[-1] - ts[0]).days
    
dataset["purchase_span_days"]= dataset["history_timestamps"].progress_apply(purchase_span)

  0%|          | 0/5559 [00:00<?, ?it/s]

100%|██████████| 5559/5559 [00:00<00:00, 12577.81it/s]


In [14]:
def ensure_list(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

def build_profile_text(row):
    categories= ensure_list(row["preferred_categories"])
    liked= ensure_list(row["liked_products"])
    recent= ensure_list(row["recent_products"])
    categories= [str(c) for c in categories if c is not None]
    liked= [str(x) for x in liked if x is not None]
    recent= [str(x) for x in recent if x is not None]
    category_text= (
        "\n- "+"\n- ".join(categories)
        if categories else
        "\n- Unknown"
    )
    liked_text= (
        "\n- "+"\n- ".join(liked)
        if liked else
        "\n- None"
    )
    recent_text= (
        "\n- "+"\n- ".join(recent)
        if recent else
        "\n- None"
    )
    return f"""
User Preference Profile

The user has primarily purchased products in these categories:
{category_text}

Products the user rated highly:
{liked_text}

Most recent purchases:
{recent_text}

The user is generally highly satisfied with previous purchases (average rating {row['average_rating']:.2f}/5).

Their purchase history spans approximately {row['purchase_span_days']} days.

Based on this profile, generate a realistic shopping query that the user might issue before purchasing their next product. The query should be natural, concise and consistent with the user's previous interests.
""".strip()

In [15]:
dataset["profile_text"]= dataset.apply(build_profile_text, axis=1)
dataset[["profile_text"]].sample(5)

,profile_text
4189,User Preference Profile\n\nThe user has primar...
3722,User Preference Profile\n\nThe user has primar...
1697,User Preference Profile\n\nThe user has primar...
972,User Preference Profile\n\nThe user has primar...
2768,User Preference Profile\n\nThe user has primar...


In [16]:
print(dataset.loc[0, "profile_text"])

User Preference Profile

The user has primarily purchased products in these categories:

- Micro SD Cards
- Portable Bluetooth Speakers
- Internal Hard Drives

Products the user rated highly:

- Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
- The AquaAudio Cubo – Waterproof Bluetooth Wireless Speaker with Strong Suction Cup
- Western Digital 2TB WD Black Performance Internal Hard Drive HDD - 7200 RPM, SATA 6 Gb/s, 64 MB Cache, 3.5" - WD2003FZEX
- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

Most recent purchases:

- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

The user is generally highly satisfied with previous purchases (average rating 5.00/5)

In [17]:
dataset.to_parquet(OUTPUT,index=False)
print("Saved to")
print(OUTPUT)

Saved to
/users/adhp511/query-conditioned-recommender/data/processed/user_preference_profiles.parquet
